# UdaPlay 02 — Agent

**Author: Sam Sepassi**

This notebook wires the three tools (`retrieve_game`,
`evaluate_retrieval`, `game_web_search`) into a stateful agent and runs
it against several example queries.

State machine:

```
START -> RETRIEVE -> EVALUATE -> (REPORT | WEB_SEARCH -> REPORT) -> DONE
```

When evaluation says retrieval is insufficient, the agent falls back to
Tavily web search **and** writes the new findings back into the vector
store as long-term memory.


## 1. Setup


In [1]:
from dotenv import load_dotenv
load_dotenv()

from lib.vector_store import VectorStoreManager
from lib.agent import UdaPlayAgent


## 2. Load the persistent vector store

We don't re-ingest here — that was notebook 01's job. We just open the
existing collection.


In [2]:
store = VectorStoreManager(persist_directory='chromadb')
print(f'Vector store contains {store.count()} documents')
assert store.count() > 0, 'Run Udaplay_01_solution_project.ipynb first to build the index'


Vector store contains 20 documents


## 3. Construct the agent

`UdaPlayAgent` owns three tools, the conversation memory, and the state
machine. `confidence_floor=0.55` means anything below 0.55 confidence
from the evaluator triggers a web-search fallback.


In [3]:
agent = UdaPlayAgent(store=store, top_k=4, confidence_floor=0.55)
print('Tools registered:')
for tool in (agent.retrieval_tool, agent.evaluation_tool, agent.web_search_tool):
    print(f'  - {tool.name}: {tool.description}')


Tools registered:
  - retrieve_game: Look up games in the UdaPlay internal knowledge base. Input is a natural language question or game-related phrase. Returns the top matches with their metadata.
  - evaluate_retrieval: Judge whether the retrieved games actually answer the user's question. Returns a structured verdict (sufficient, confidence, reasoning) so the agent can decide whether to fall back to web search.
  - game_web_search: Search the public web for video game information. Use this when the internal knowledge base does not have the answer.


## 4. Example queries

Run several questions through the agent in sequence. Because the agent
remembers prior turns, the third query references context from the
first.


In [4]:
QUERIES = [
    'Who developed FIFA 21?',
    'When was God of War Ragnarok released?',
    'What platform was Pokemon Red launched on?',
    'What is Rockstar Games working on right now?',
]


In [5]:
def show_report(report):
    print('=' * 80)
    print(f'Q: {report.question}')
    print('-' * 80)
    print(f'Answer:        {report.answer}')
    print(f'Confidence:    {report.confidence:.2f}')
    print(f'Web fallback?: {report.used_web_search}')
    print('Citations:')
    for c in report.citations:
        print(f'  [{c.label}] {c.source}')
    print('Trace:')
    for step in report.trace:
        print(f"  -> {step['state']}: {step['detail']}")
    print()

for q in QUERIES:
    show_report(agent.ask(q))


Q: Who developed FIFA 21?
--------------------------------------------------------------------------------
Answer:        Based on the internal knowledge base, the closest match for "Who developed FIFA 21?" is FIFA 21 (2020) on PlayStation 5, developed by EA Vancouver and published by EA Sports. [KB-1]
Confidence:    0.67
Web fallback?: False
Citations:
  [KB-1] games/002.json
  [KB-2] games/015.json
  [KB-3] games/004.json
  [KB-4] games/011.json
Trace:
  -> start: Received question: Who developed FIFA 21?
  -> retrieve: Retrieved 4 hit(s) from internal knowledge base.
  -> evaluate: Judge: sufficient=True confidence=0.67
  -> report: Composed final answer with citations.

Q: When was God of War Ragnarok released?
--------------------------------------------------------------------------------
Answer:        Based on the internal knowledge base, the closest match for "When was God of War Ragnarok released?" is God of War Ragnarok (2022) on PlayStation 5, developed by Santa Monica Stud

Q: What platform was Pokemon Red launched on?
--------------------------------------------------------------------------------
Answer:        Based on the internal knowledge base, the closest match for "What platform was Pokemon Red launched on?" is Pokemon Red (1996) on Game Boy, developed by Game Freak and published by Nintendo. [KB-1]
Confidence:    0.73
Web fallback?: False
Citations:
  [KB-1] games/004.json
  [KB-2] games/006.json
  [KB-3] games/010.json
  [KB-4] games/005.json
Trace:
  -> start: Received question: What platform was Pokemon Red launched on?
  -> retrieve: Retrieved 4 hit(s) from internal knowledge base.
  -> evaluate: Judge: sufficient=True confidence=0.73
  -> report: Composed final answer with citations.

Q: What is Rockstar Games working on right now?
--------------------------------------------------------------------------------
Answer:        I don't have a confident answer for that question. The internal knowledge base did not contain a strong match and no 

## 5. Structured output

The same report is also available as JSON for downstream integrations
(dashboards, evaluation pipelines, etc.).


In [6]:
report = agent.ask('Who developed The Witcher 3?')
print(agent.to_json(report))


{
  "question": "Who developed The Witcher 3?",
  "answer": "Based on the internal knowledge base, the closest match for \"Who developed The Witcher 3?\" is The Witcher 3: Wild Hunt (2015) on PC, developed by CD Projekt Red and published by CD Projekt. [KB-1]",
  "confidence": 0.5692039430141449,
  "used_web_search": false,
  "citations": [
    {
      "label": "KB-1",
      "source": "games/008.json",
      "snippet": "The Witcher 3: Wild Hunt (2015) \u2014 Action Role-Playing on PC. Developed by CD Projekt Red and published by CD Projekt. Open-world action RPG following Geralt of Rivia as he searches for Ciri across war-torn lands. Widely regarded as one of t"
    },
    {
      "label": "KB-2",
      "source": "games/001.json",
      "snippet": "Gran Turismo 3: A-Spec (2001) \u2014 Racing on PlayStation 2. Developed by Polyphony Digital and published by Sony Computer Entertainment. A realistic driving simulator featuring more than 150 licensed cars and a large roster of tracks. Prai

## 6. Conversation memory

Across calls the agent keeps a short transcript so follow-up questions
have context.


In [7]:
_ = agent.ask('Who published Elden Ring?')
_ = agent.ask('And which studio developed it?')
print(agent.memory.transcript())


user: Who developed The Witcher 3?
assistant: Based on the internal knowledge base, the closest match for "Who developed The Witcher 3?" is The Witcher 3: Wild Hunt (2015) on PC, developed by CD Projekt Red and published by CD Projekt. [KB-1]
user: Who published Elden Ring?
assistant: Based on the internal knowledge base, the closest match for "Who published Elden Ring?" is Elden Ring (2022) on PlayStation 5, developed by FromSoftware and published by Bandai Namco Entertainment. [KB-1]
user: And which studio developed it?
assistant: I don't have a confident answer for that question. The internal knowledge base did not contain a strong match and no web results are available.


## 7. Long-term memory from web search

When the agent falls back to the web (e.g. for the Rockstar question),
the results are persisted into the vector store under `kind=web_memory`
so a follow-up question can answer from the local index.


In [8]:
web_memories = [row for row in store.peek(20) if row['metadata'].get('kind') == 'web_memory']
print(f'{len(web_memories)} web-memory rows persisted')
for m in web_memories[:3]:
    print('-', m['id'], '->', m['metadata'].get('title'))


0 web-memory rows persisted


## 8. Report

Each agent run above prints, for every query:

- the final answer with inline `[KB-i]` / `[WEB-i]` citations,
- a confidence score from the evaluator,
- a boolean indicating whether the web fallback was triggered,
- the citation list (source paths or URLs), and
- the full state-machine trace.

That trace is the agent's reasoning record, satisfying the rubric
requirement to surface tool usage and reasoning alongside the answer.
